# Interactive Risk Game

Play Risk against any of pyrisk's built-in AIs, Gemini LLM, or the fine-tuned Qwen model.
The board renders as a matplotlib map after each phase, and you make decisions via `input()`.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import sys, os, random

# Path setup
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'pyrisk_vendor'))
sys.path.insert(0, PROJECT_ROOT)

from game import Game
from world import CONNECT, MAP, KEY, AREAS
from ai.stupid import StupidAI
from ai.better import BetterAI
from ai.al import AlAI
from ai.chron import ChronAI
from llm_player.hybrid_player import HybridPlayer
from human_player.human_player import HumanAI
from playing.risk_map import draw_board

# ── Choose opponent ───────────────────────────────────────────────────
OPPONENTS = {
    '1': ('StupidAI',  StupidAI,  None),
    '2': ('BetterAI',  BetterAI,  None),
    '3': ('AlAI',      AlAI,      None),
    '4': ('ChronAI',   ChronAI,   None),
    '5': ('Gemini LLM', HybridPlayer, None),
    '6': ('Qwen GRPO v3', HybridPlayer, 'victoria-kp/risk-qwen-grpo-v3'),
}

print('Choose your opponent:')
for k, (name, _, _) in OPPONENTS.items():
    print(f'  {k}. {name}')

choice = input('Enter number [1-6]: ').strip()
opp_name, opp_class, model_path = OPPONENTS.get(choice, ('StupidAI', StupidAI, None))
print(f'\nOpponent: {opp_name}')

# Set model path: Gemini uses GOOGLE_API_KEY (no RISK_MODEL_PATH),
# Qwen uses the HuggingFace model path
if model_path:
    os.environ['RISK_MODEL_PATH'] = model_path
elif choice == '5':
    # Gemini: clear RISK_MODEL_PATH so it falls back to API
    os.environ.pop('RISK_MODEL_PATH', None)

# ── Create game ───────────────────────────────────────────────────────
game = Game(curses=False, connect=CONNECT, cmap=MAP, ckey=KEY, areas=AREAS)
game.add_player('Human', HumanAI)
game.add_player('OPP',   opp_class)
game.add_player('FILL',  StupidAI)

# Wire render callback so HumanAI draws the map
human_ai = game.players['Human'].ai
human_ai.render_fn = lambda highlight=None: draw_board(
    game, current_player='Human', highlight=highlight
)

print(f'Game ready: Human vs {opp_name} (+ StupidAI filler)')
print('Run the next cell to start playing.')

In [ ]:
# ── Test: verify map renders before playing ──────────────────────────
# If you see the Risk map below, display is working.
# If not: Restart kernel, then run cells in order.
import matplotlib.pyplot as plt
plt.ioff()

print("Testing draw_board...")
draw_board(game, current_player='Human')
print("If you see the Risk map above, you're good. Run the next cell to play.")

In [ ]:
# ── Play ──────────────────────────────────────────────────────────────
# This blocks until the game ends, interleaving map renders with input() prompts.
# AI opponents play their turns silently between your turns.

winner = game.play()
print(f'\n{"=" * 40}')
print(f'GAME OVER — Winner: {winner}')
print(f'{"=" * 40}')

In [ ]:
# ── Results ───────────────────────────────────────────────────────────
print(f'Winner: {winner}')
print(f'Total turns: {game.turn}')
print()

for name in game.turn_order:
    p = game.players[name]
    status = 'WINNER' if p.alive else 'eliminated'
    terr_count = p.territory_count
    force_count = p.forces
    print(f'  {name:<8} {status:<12} {terr_count} territories, {force_count} forces')

print()
draw_board(game, current_player=winner)